## Data Loading
The main task in the code box below is loading the relevant log.

In [ ]:
# CHANGE HERE: Adjust the file name to load a different event log.
FILE_NAME = "datasets/Road_Traffic_Fine_Management_Process.xes"

from core.data.data_loader import load_xes_log

# Initial Log Load
try:
    file_name = FILE_NAME
    event_log_pre = load_xes_log(file_name)

except Exception as e:
    raise RuntimeError(f"Failed to load event log: {e}")

## Log Transformations
Experiment with log transformations in the code box below.

In [ ]:
# CHANGE HERE: Any potential enrichments besides activity folding or activity subsetting:
event_log_pre = event_log_pre.sort_values(
    ["case:concept:name", "time:timestamp"]
).reset_index(drop=True)

# Extend log with additional attributes
event_log_pre['amount::latest'] = event_log_pre.groupby('case:concept:name')['amount'].ffill()

event_log_pre['expense::sum'] = event_log_pre.groupby('case:concept:name')['expense'].transform(lambda x: x.fillna(0).cumsum())

event_log_pre['paymentAmount::sum'] = event_log_pre.groupby('case:concept:name')['paymentAmount'].transform(lambda x: x.fillna(0).cumsum())

event_log_pre['outstanding_balance'] = event_log_pre.apply(
    lambda row: round(row['amount::latest'] + row['expense::sum'] - row['paymentAmount::sum'], 2), 
    axis=1)

# Classify activities which shall be unfolded
partial_payment_mask = (event_log_pre["concept:name"] == "Payment") & (event_log_pre["outstanding_balance"] > 0)
full_payment_mask = (event_log_pre["concept:name"] == "Payment") & (event_log_pre["outstanding_balance"] <= 0)

prefecture_dismissed = (event_log_pre["concept:name"] == "Send Appeal to Prefecture") & (event_log_pre["dismissal"] == "#")
judge_dismissed = (event_log_pre["concept:name"] == "Appeal to Judge") & (event_log_pre["dismissal"] == "G")

# Create a copy of the original log to modify activity names for unfolding based on the above classifications
event_log = event_log_pre.copy()
event_log.loc[partial_payment_mask, "concept:name"] = "Partial Payment"
event_log.loc[full_payment_mask, "concept:name"] = "Full Payment"
event_log.loc[prefecture_dismissed, "concept:name"] = "Prefecture Dismissed"
event_log.loc[judge_dismissed, "concept:name"] = "Judge Dismissed"
event_log.head()

## Running the App
Starting of UI app with transformed data within the notebook. No changes should be necessary here.

In [ ]:
import importlib
import dapp_fact
importlib.reload(dapp_fact)

app = dapp_fact.create_app(event_log)
app.run()